# Reproducing All Results — Customer Churn Prediction in Waste Banks (XGBoost + SMOTE)

This notebook reproduces **every quantitative result** reported in the manuscript, using the de-identified dataset in this repository.

**Contents**
1. Setup and data loading
2. Main model performance (Table 1)
3. Baseline model comparison (Table 2)
4. Churn-definition sensitivity (Figures 7–8)
5. Hyperparameter optimization (Table 3)
6. Bootstrap confidence intervals and significance tests (Section 3.8)
7. Temporal forward-chaining validation (Section 3.9)
8. Permutation importance (Figure 10)
9. Probability calibration (Figure 11)
10. Fairness/subgroup analysis and multicollinearity (Section 3.9)
11. Nested cross-validation (Section 3.9)
12. Computational cost (Section 3.9)

**Requirements:** Python 3.10+, `pandas`, `numpy`, `scikit-learn`, `imbalanced-learn`, `xgboost`, `lightgbm`, `scipy`, `matplotlib`, `openpyxl`.
All analyses use fixed seeds (`random_state=42`) and are deterministic.

## 1. Setup and data loading

Set `PROJ` to the folder that contains `01_raw_data/`, `02_clean_data/`, and `03_experiment_split/`.

* **Local (Windows):** `PROJ = r'D:\path\to\repo'` — the `r'...'` prefix is required.
* **Local (macOS/Linux):** `PROJ = '/path/to/repo'`
* **Google Colab:** run the optional upload cell below, then use `PROJ = '/content/pkg'`.

In [ ]:
# --- Google Colab helper: just RUN this cell. Does nothing outside Colab. ---
# Upload EITHER an archive of the repository (.zip or .rar), OR the individual .xlsx files.
# You can re-run this cell as many times as you like to add more files.
import os, sys, shutil, subprocess, zipfile
try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def _route(fname):
    if 'dataset_churn' in fname:   return '02_clean_data'
    if 'BUKU BESAR' in fname:      return '01_raw_data'
    if 'final_no_alamat' in fname: return '03_experiment_split'
    return None

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'xgboost', 'lightgbm', 'imbalanced-learn', 'openpyxl'])
    BASE = '/content/pkg'
    for sub in ['01_raw_data', '02_clean_data', '03_experiment_split']:
        os.makedirs(f'{BASE}/{sub}', exist_ok=True)

    print('Upload a .zip / .rar of the repository, or the .xlsx files (re-run to add more).')
    uploaded = _colab_files.upload()

    for fn in uploaded:
        src, low = f'/content/{fn}', fn.lower()
        if low.endswith(('.zip', '.rar')):
            outdir = '/content/_arch'; os.makedirs(outdir, exist_ok=True)
            if low.endswith('.zip'):
                with zipfile.ZipFile(src) as z:
                    z.extractall(outdir)
            else:
                subprocess.run('apt-get -qq install -y unrar', shell=True,
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                r = subprocess.run(['unrar', 'x', '-o+', '-inul', src, outdir + '/'])
                if r.returncode != 0:
                    print('  Could not extract the .rar. Please upload a .zip or the .xlsx files instead.')
            found = 0
            for root, _d, fs in os.walk(outdir):
                for f in fs:
                    sub = _route(f)
                    if f.endswith('.xlsx') and sub:
                        shutil.copy(os.path.join(root, f), f'{BASE}/{sub}/{f}'); found += 1
            print(f'  extracted {found} data file(s) from {fn}')
            os.remove(src)
        else:
            sub = _route(fn)
            if sub:
                shutil.move(src, f'{BASE}/{sub}/{fn}')
            else:
                print('  (ignored, unrecognised file):', fn)

    print()
    need = {'01_raw_data': 1, '02_clean_data': 1, '03_experiment_split': 2}
    ok = True
    for sub, k in need.items():
        have = [f for f in os.listdir(f'{BASE}/{sub}') if f.endswith('.xlsx')]
        print(f'  {sub}: {have}')
        if len(have) < k:
            ok = False
    print()
    print('Set in the NEXT cell:   PROJ = "/content/pkg"')
    if not ok:
        print('NOTE: some files are still missing - re-run this cell and upload the rest.')
else:
    print('Not running on Colab - skip this cell and set PROJ to your local repository folder.')

In [ ]:
import os, time, warnings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    matthews_corrcoef, balanced_accuracy_score, cohen_kappa_score, brier_score_loss,
    confusion_matrix, roc_curve, precision_recall_curve, average_precision_score)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')

# =====================================================================
# >>> EDIT ONLY THIS LINE <<<
PROJ = '.'
# =====================================================================

SP  = f'{PROJ}/03_experiment_split'
CLN = f'{PROJ}/02_clean_data'
RAW = f'{PROJ}/01_raw_data/BUKU BESAR BANK SAMPAH LH.xlsx'
assert os.path.isdir(SP), f'Not found: {SP}\nFix PROJ above (Windows needs r"...").'

SEED = 42
FEATS = ['Recency','Frequency','Monetary','TotalBerat','Jenis Kelamin','Bidang']   # 6 predictors
NUM   = ['Recency','Frequency','Monetary','TotalBerat']

trn = pd.read_excel(f'{SP}/train_final_no_alamat.xlsx')
tst = pd.read_excel(f'{SP}/test_final_no_alamat.xlsx')
Xtr = pd.get_dummies(trn[FEATS]); Xte = pd.get_dummies(tst[FEATS])
Xtr, Xte = Xtr.align(Xte, join='left', axis=1, fill_value=0)
ytr, yte = trn['ChurnFuture'].values, tst['ChurnFuture'].values

# SMOTE is applied to the TRAINING data only
Xs, ys = SMOTE(random_state=SEED).fit_resample(Xtr, ytr)
scaler = StandardScaler().fit(Xs); Xs_s, Xte_s = scaler.transform(Xs), scaler.transform(Xte)

def make_xgb():
    return XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         eval_metric='logloss', random_state=SEED)

print(f'Train {Xtr.shape} -> after SMOTE {Xs.shape} | Test {Xte.shape} (churn={int(yte.sum())})')

## 2. Main model performance — Table 1

In [ ]:
xgb = make_xgb().fit(Xs, ys)
p   = xgb.predict_proba(Xte)[:,1]
pred = (p >= 0.5).astype(int)

print('Accuracy ', round(accuracy_score(yte,pred),4))
print('Precision', round(precision_score(yte,pred),4))
print('Recall   ', round(recall_score(yte,pred),4))
print('F1-score ', round(f1_score(yte,pred),4))
print('ROC AUC  ', round(roc_auc_score(yte,p),4))
print('MCC      ', round(matthews_corrcoef(yte,pred),4))
print('Bal. Acc ', round(balanced_accuracy_score(yte,pred),4))
print("Kappa    ", round(cohen_kappa_score(yte,pred),4))
print('Brier    ', round(brier_score_loss(yte,p),4))
print('\nConfusion matrix [[TN FP],[FN TP]]:\n', confusion_matrix(yte,pred))

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,4.2))
fpr,tpr,_ = roc_curve(yte,p)
ax[0].plot(fpr,tpr,lw=2,label=f'AUC = {roc_auc_score(yte,p):.3f}'); ax[0].plot([0,1],[0,1],'--',color='orange')
ax[0].set_xlabel('False Positive Rate'); ax[0].set_ylabel('True Positive Rate'); ax[0].set_title('ROC Curve'); ax[0].legend()
pr,rc,_ = precision_recall_curve(yte,p); ap = average_precision_score(yte,p)
ax[1].plot(rc,pr,lw=2,color='#C44E52',label=f'AP = {ap:.3f}'); ax[1].axhline(yte.mean(),ls='--',color='grey',label=f'prevalence = {yte.mean():.3f}')
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision'); ax[1].set_title('Precision-Recall Curve'); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. Baseline model comparison — Table 2

All models share the identical customer-level split, SMOTE-balanced training data and encoding.
Logistic Regression and SVM would additionally require standardized features; the four reported baselines are used here.

In [ ]:
models = {
 'XGBoost':             (make_xgb(), False),
 'LightGBM':            (LGBMClassifier(n_estimators=250,max_depth=4,learning_rate=0.05,subsample=0.8,
                                        colsample_bytree=0.8,random_state=SEED,verbose=-1), False),
 'Random Forest':       (RandomForestClassifier(n_estimators=300,random_state=SEED,n_jobs=-1), False),
 'Logistic Regression': (LogisticRegression(max_iter=2000,random_state=SEED), True),
 'Decision Tree':       (DecisionTreeClassifier(max_depth=6,random_state=SEED), False),
}
P, PRED, rows = {}, {}, []
for name,(m,scale) in models.items():
    A,B = (Xs_s,Xte_s) if scale else (Xs,Xte)
    m.fit(A,ys); pp = m.predict_proba(B)[:,1]; pr_ = (pp>=0.5).astype(int)
    P[name], PRED[name] = pp, pr_
    rows.append({'Model':name,'Acc.':round(accuracy_score(yte,pr_),3),'Prec.':round(precision_score(yte,pr_),3),
                 'Recall':round(recall_score(yte,pr_),3),'F1':round(f1_score(yte,pr_),3),
                 'ROC AUC':round(roc_auc_score(yte,pp),4),'MCC':round(matthews_corrcoef(yte,pr_),3),
                 'Bal. Acc.':round(balanced_accuracy_score(yte,pr_),3),'Kappa':round(cohen_kappa_score(yte,pr_),3)})
pd.DataFrame(rows).sort_values('ROC AUC',ascending=False).reset_index(drop=True)

## 4. Churn-definition sensitivity — Figures 7–8

Uses the de-identified transaction log to (a) measure inter-transaction intervals and
(b) rebuild the churn label under 30/90/180-day inactivity windows.

In [ ]:
def load_deposits():
    tx = pd.read_excel(RAW, sheet_name='Form Responses 1'); tx.columns = [c.strip() for c in tx.columns]
    tx = tx.rename(columns={'Timestamp':'Tanggal','Jenis Kegiatan':'Kegiatan',
                            'Total Tabungan (Rp.)':'Total','Berat (kg)':'Berat'})
    tx['NoRek'] = tx['Nomor Rekening']
    tx['Tanggal'] = pd.to_datetime(tx['Tanggal'], errors='coerce')
    tx = tx.dropna(subset=['Tanggal'])
    d = tx[tx['Kegiatan']=='Penyetoran Sampah'].copy()
    for c in ['Total','Berat']: d[c] = pd.to_numeric(d[c], errors='coerce')
    return d.dropna(subset=['NoRek','Total','Berat'])

def build_snapshots(dep, future_days, snap_dates):
    out=[]
    for snap in snap_dates:
        hist = dep[dep['Tanggal']<=snap]
        recent = hist[hist['Tanggal']>=snap-pd.Timedelta(days=90)]['NoRek'].unique()
        hist = hist[hist['NoRek'].isin(recent)]
        if len(hist)==0: continue
        fut = dep[(dep['Tanggal']>snap)&(dep['Tanggal']<=snap+pd.Timedelta(days=future_days))]
        g = hist.groupby('NoRek').agg({'Tanggal':[lambda x:(snap-x.max()).days,'count'],'Total':'sum','Berat':['sum','mean']})
        g.columns=['Recency','Frequency','Monetary','TotalBerat','AvgBerat']; g=g.reset_index()
        g['ChurnFuture']=np.where(g['NoRek'].isin(fut['NoRek'].unique()),0,1); out.append(g)
    return pd.concat(out, ignore_index=True)

dep = load_deposits().sort_values(['NoRek','Tanggal'])
gaps = dep.groupby('NoRek')['Tanggal'].diff().dt.days.dropna(); gaps = gaps[gaps>=0]
print(f'Median interval {gaps.median():.0f} d | 90th pct {np.percentile(gaps,90):.0f} d')
print(f'<=30d {(gaps<=30).mean()*100:.1f}% | <=90d {(gaps<=90).mean()*100:.1f}% | <=180d {(gaps<=180).mean()*100:.1f}%')

In [ ]:
smin,smax = dep['Tanggal'].min(), dep['Tanggal'].max()
snaps = pd.date_range(start=smin+pd.DateOffset(months=6), end=smax-pd.DateOffset(months=6), freq='ME')
res=[]
for d_ in [30,90,180]:
    ds = build_snapshots(dep, d_, snaps)
    ids = ds['NoRek'].unique(); a,b = train_test_split(ids, test_size=0.2, random_state=SEED)
    A,Bd = ds[ds['NoRek'].isin(a)], ds[ds['NoRek'].isin(b)]
    Xa,ya = SMOTE(random_state=SEED).fit_resample(A[NUM], A['ChurnFuture'])
    m = make_xgb().fit(Xa,ya); pp = m.predict_proba(Bd[NUM])[:,1]
    res.append({'window (days)':d_, 'churn rate':round(ds['ChurnFuture'].mean(),3),
                'test ROC AUC':round(roc_auc_score(Bd['ChurnFuture'],pp),3),
                'test F1':round(f1_score(Bd['ChurnFuture'],(pp>=0.5).astype(int)),3)})
pd.DataFrame(res)

## 5. Hyperparameter optimization — Table 3

SMOTE is placed **inside** the cross-validation pipeline so oversampling is applied to training folds only.

In [ ]:
space = {'clf__n_estimators':[150,250,350,500],'clf__max_depth':[3,4,5,6],
         'clf__learning_rate':[0.01,0.03,0.05,0.1],'clf__subsample':[0.7,0.8,0.9,1.0],
         'clf__colsample_bytree':[0.7,0.8,0.9,1.0],'clf__min_child_weight':[1,3,5],'clf__gamma':[0,0.1,0.3]}
def pipe(): return ImbPipeline([('smote',SMOTE(random_state=SEED)),('clf',XGBClassifier(eval_metric='logloss',random_state=SEED))])
rs = RandomizedSearchCV(pipe(), space, n_iter=25, scoring='roc_auc',
                        cv=StratifiedKFold(3,shuffle=True,random_state=SEED), random_state=SEED, n_jobs=-1).fit(Xtr,ytr)
p_t = rs.predict_proba(Xte)[:,1]
cv_adopt = cross_val_score(ImbPipeline([('smote',SMOTE(random_state=SEED)),('clf',make_xgb())]),
                           Xtr,ytr,cv=StratifiedKFold(5,shuffle=True,random_state=SEED),scoring='roc_auc')
print('Search best params :', {k.replace('clf__',''):v for k,v in rs.best_params_.items()})
print(f'Randomized search  : CV AUC {rs.best_score_:.3f} | test AUC {roc_auc_score(yte,p_t):.3f} | test F1 {f1_score(yte,(p_t>=0.5).astype(int)):.3f}')
print(f'Adopted (manuscript): CV AUC {cv_adopt.mean():.3f} (+/- {cv_adopt.std():.3f}) | test AUC {roc_auc_score(yte,p):.3f} | test F1 {f1_score(yte,pred):.3f}')

## 6. Bootstrap confidence intervals and significance tests — Section 3.8

In [ ]:
rng = np.random.RandomState(SEED); n = len(yte); idx = np.arange(n); B = 2000
def all_metrics(y,pp,pr_):
    return dict(Accuracy=accuracy_score(y,pr_),Precision=precision_score(y,pr_),Recall=recall_score(y,pr_),
                F1=f1_score(y,pr_),ROC_AUC=roc_auc_score(y,pp),MCC=matthews_corrcoef(y,pr_),
                BalancedAcc=balanced_accuracy_score(y,pr_),Kappa=cohen_kappa_score(y,pr_),Brier=brier_score_loss(y,pp))
point = all_metrics(yte,p,pred); boot = {k:[] for k in point}
for _ in range(B):
    s_ = rng.choice(idx,n,replace=True)
    if len(np.unique(yte[s_]))<2: continue
    m_ = all_metrics(yte[s_],p[s_],pred[s_])
    for k in m_: boot[k].append(m_[k])
print('Metric        estimate   95% CI')
for k in point:
    print(f'{k:<13} {point[k]:.4f}   [{np.percentile(boot[k],2.5):.4f}, {np.percentile(boot[k],97.5):.4f}]')

In [ ]:
print('\nPairwise comparison against XGBoost:')
for b in ['LightGBM','Random Forest','Logistic Regression','Decision Tree']:
    diffs=[]
    for _ in range(B):
        s_ = rng.choice(idx,n,replace=True)
        if len(np.unique(yte[s_]))<2: continue
        diffs.append(roc_auc_score(yte[s_],p[s_]) - roc_auc_score(yte[s_],P[b][s_]))
    diffs=np.array(diffs); pv = 2*min((diffs<=0).mean(),(diffs>=0).mean())
    b01=np.sum((pred==yte)&(PRED[b]!=yte)); b10=np.sum((pred!=yte)&(PRED[b]==yte))
    mcp = stats.chi2.sf((abs(b01-b10)-1)**2/(b01+b10),1) if (b01+b10)>0 else 1.0
    print(f'  vs {b:<20} dAUC={diffs.mean():+.4f}  95% CI [{np.percentile(diffs,2.5):+.4f}, {np.percentile(diffs,97.5):+.4f}]  bootstrap p={pv:.3f}  McNemar p={mcp:.3f}')

## 7. Temporal forward-chaining validation — Section 3.9

In [ ]:
full = pd.read_excel(f'{CLN}/dataset_churn_cleaned_bidang.xlsx')
full['SnapshotDate'] = pd.to_datetime(full['SnapshotDate'])
months = sorted(full['SnapshotDate'].unique()); aucs=[]; f1s=[]
for i in range(3,len(months)-1):
    tr_ = full[full['SnapshotDate']<=months[i]]; te_ = full[full['SnapshotDate']==months[i+1]]
    if te_['ChurnFuture'].nunique()<2 or len(te_)<30: continue
    Xa = pd.get_dummies(tr_[FEATS]); Xb = pd.get_dummies(te_[FEATS]); Xa,Xb = Xa.align(Xb,join='left',axis=1,fill_value=0)
    Xa2,ya2 = SMOTE(random_state=SEED).fit_resample(Xa, tr_['ChurnFuture'].values)
    m = make_xgb().fit(Xa2,ya2); pb = m.predict_proba(Xb)[:,1]
    aucs.append(roc_auc_score(te_['ChurnFuture'].values,pb)); f1s.append(f1_score(te_['ChurnFuture'].values,(pb>=0.5).astype(int)))
print(f'Forward-chaining ROC AUC: {np.mean(aucs):.3f} +/- {np.std(aucs):.3f} | mean F1 {np.mean(f1s):.3f} ({len(aucs)} folds)')

## 8. Permutation importance — Figure 10

In [ ]:
pi = permutation_importance(xgb, Xte, yte, n_repeats=20, random_state=SEED, scoring='roc_auc', n_jobs=-1)
def base_feat(c):
    for b in ['Jenis Kelamin','Bidang']:
        if c.startswith(b): return b
    return c
agg = {}
for c,v in zip(Xte.columns, pi.importances_mean): agg[base_feat(c)] = agg.get(base_feat(c),0)+v
order = sorted(agg.items(), key=lambda x: x[1])
for k,v in reversed(order): print(f'{k:<16} {v:.4f}')
plt.figure(figsize=(6.2,3.4)); plt.barh([k for k,_ in order],[v for _,v in order],color='#4C72B0')
plt.xlabel('Permutation importance (mean AUC drop)'); plt.title('Permutation Feature Importance'); plt.tight_layout(); plt.show()

## 9. Probability calibration — Figure 11

In [ ]:
print('Brier score:', round(brier_score_loss(yte,p),4))
frac, meanp = calibration_curve(yte, p, n_bins=10, strategy='quantile')
plt.figure(figsize=(4.6,4.6)); plt.plot([0,1],[0,1],'--',color='grey',label='Perfect calibration')
plt.plot(meanp, frac, 'o-', color='#C44E52', label=f'XGBoost (Brier={brier_score_loss(yte,p):.3f})')
plt.xlabel('Mean predicted probability'); plt.ylabel('Observed churn frequency')
plt.title('Calibration Curve'); plt.legend(); plt.tight_layout(); plt.show()

## 10. Fairness / subgroup analysis and multicollinearity — Section 3.9

In [ ]:
print('Subgroup performance by gender:')
for g in tst['Jenis Kelamin'].unique():
    mask = (tst['Jenis Kelamin']==g).values
    if mask.sum()<20 or len(np.unique(yte[mask]))<2: continue
    print(f'  {g:<12} n={mask.sum():<5} churn={yte[mask].mean():.3f} AUC={roc_auc_score(yte[mask],p[mask]):.3f} '
          f'recall={recall_score(yte[mask],pred[mask]):.3f} precision={precision_score(yte[mask],pred[mask],zero_division=0):.3f}')

print('\nSubgroup performance by department (n>=50):')
for g in tst['Bidang'].unique():
    mask = (tst['Bidang']==g).values
    if mask.sum()<50 or len(np.unique(yte[mask]))<2: continue
    print(f'  {g:<18} n={mask.sum():<5} AUC={roc_auc_score(yte[mask],p[mask]):.3f}')

from numpy.linalg import inv
corr = np.corrcoef(trn[NUM].astype(float).values.T); vif = np.diag(inv(corr))
print('\nVariance inflation factors:', {NUM[i]: round(float(vif[i]),3) for i in range(len(NUM))})

## 11. Nested cross-validation — Section 3.9

Confirms that tuning (performed on training data only) does not introduce optimistic bias.

In [ ]:
space_n = {'clf__n_estimators':[150,250,350],'clf__max_depth':[3,4,5],'clf__learning_rate':[0.03,0.05,0.1],
           'clf__subsample':[0.8,0.9,1.0],'clf__colsample_bytree':[0.8,0.9,1.0]}
outer = StratifiedKFold(5, shuffle=True, random_state=SEED); nested=[]
for tr_i, va_i in outer.split(Xtr, ytr):
    inner = RandomizedSearchCV(pipe(), space_n, n_iter=8, scoring='roc_auc',
                               cv=StratifiedKFold(3,shuffle=True,random_state=1), random_state=SEED, n_jobs=-1)
    inner.fit(Xtr.iloc[tr_i], ytr[tr_i])
    nested.append(roc_auc_score(ytr[va_i], inner.predict_proba(Xtr.iloc[va_i])[:,1]))
print(f'Nested CV ROC AUC: {np.mean(nested):.4f} +/- {np.std(nested):.4f}')
print(f'Held-out test ROC AUC: {roc_auc_score(yte,p):.4f}')

## 12. Computational cost — Section 3.9

Absolute times depend on hardware; report the values obtained on your own machine.

In [ ]:
t0=time.time(); _m = make_xgb().fit(Xs,ys); train_s = time.time()-t0
t1=time.time(); _  = _m.predict_proba(Xte);            infer_ms = 1000*(time.time()-t1)/len(yte)
print(f'Training time : {train_s:.2f} s  (n_train={len(ys)})')
print(f'Inference time: {infer_ms:.4f} ms per member  (n_test={len(yte)})')
print(f'Model size    : {make_xgb().get_params()["n_estimators"]} trees, max_depth={make_xgb().get_params()["max_depth"]}')

## Expected results (for verification)

| Quantity | Expected |
|---|---|
| ROC AUC (test) | 0.8985 — 95% CI [0.881, 0.914] |
| Accuracy / Precision / Recall / F1 | 0.839 / 0.709 / 0.751 / 0.729 |
| MCC / Balanced Acc / Kappa | 0.615 / 0.813 / 0.615 |
| Brier score | 0.116 |
| Baseline ranking (ROC AUC) | XGBoost 0.8985 > LightGBM 0.8975 > RF 0.8963 > LogReg 0.8794 > DT 0.8757 |
| Significance vs LightGBM / RF | p ≈ 0.40 / 0.55 (not significant) |
| Significance vs LogReg / DT | p < 0.001 (significant) |
| Forward-chaining ROC AUC | 0.904 ± 0.031 (10 folds) |
| Nested CV ROC AUC | 0.916 ± 0.011 |
| VIF (all predictors) | < 3.5 |
| Subgroup ROC AUC (male / female) | 0.881 / 0.938 |

Training and inference **times** are hardware-dependent and will differ from the manuscript.